# Benchmark 2

Smaller benchmark

In [1]:
%pip install ollama numpy pandas openpyxl tqdm

Note: you may need to restart the kernel to use updated packages.


## Config

In [2]:
import os, json, re, time, math, hashlib, textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from typing import List, Dict, Any, Optional
from collections import Counter
from itertools import product
import ollama
import numpy as np
import pandas as pd

OLLAMA_URL  = "http://localhost:11528"
AGENT_MODEL = "gpt-oss:120b"
EMBED_MODEL = "nomic-embed-text"
CACHE_DIR  = Path("tree_cache"); TREE_FILE = CACHE_DIR / "corpus_tree.json"
QUESTIONS_FILE = "questions_updated.json"
QID = "GEN-116"                 # the one question to dissect

# the recursive-descent config; one rating call per node, fast
STRATEGY      = "cluster"      # virtual subfolders so a wide folder becomes <= MAX_BRANCH groups, never 30 at once
GROUP_SUMMARY = "llm"          # real semantic summary per group, a far better routing signal than a name list
MAX_BRANCH    = 6              # options per decision; vgroups keep every choice this narrow
RELEVANT_AT   = 0.50           # a child counts as relevant at/above this; ALL children relevant -> take the whole unit
BUDGET        = 80             # safety cap on total nodes visited
MAX_STEPS     = 24
QUERY_EXPANSION = False         # off; the raw question keeps relational meaning that keyword expansion loses
THINKING      = False           # rating runs think-off for clean json
MAX_EVIDENCE  = 50
MAX_FILES     = 2              # synthesis: hard cap on distinct files referenced
LOOKUP_FILES  = 2              # lookup: single-file lock (set to 2 to also allow one runner-up file)
SCORE_MODE    = "judge"
KEEP_ALIVE    = "30m"
print(f"agent {AGENT_MODEL}; question {QID}; recursive descent, take whole unit when all children >= {RELEVANT_AT}")

# ---- combined-pipeline config ----
MAX_QUESTIONS  = 19                                   # only the first 100 questions
QUESTIONS_FILE = "questions_updated.json"
ANSWERS_FILE   = "treerag_answers_2.json"              # step 2 output (new file)
QMS_FILE       = "qms_answers.json"                    # existing baseline answers
ANS_CACHE      = Path("answers_cache_2"); ANS_CACHE.mkdir(exist_ok=True)
ANS_STORE      = ANS_CACHE / "answers.json"            # crash-safe per-question checkpoint

OUT_DIR        = Path("second_benchmark"); OUT_DIR.mkdir(exist_ok=True)
REPORT_JSON    = OUT_DIR / "benchmark_report_2.json"
BYTYPE_JSON    = OUT_DIR / "benchmark_by_type_2.json"
REPORT_CSV     = OUT_DIR / "benchmark_per_question_2.csv"
JUDGE_MODEL    = AGENT_MODEL
ACC_GAP_FLAG   = 0.5
print("combined pipeline: answer first", MAX_QUESTIONS, "-> judge vs qms -> report into", str(OUT_DIR))


agent gpt-oss:120b; question GEN-116; recursive descent, take whole unit when all children >= 0.5
combined pipeline: answer first 19 -> judge vs qms -> report into second_benchmark


## Ollama connection + agent llm/embed + judge caller

In [3]:
client = ollama.Client(host=OLLAMA_URL, timeout=600)

def _model_names(r):
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out=[]
    for m in raw:
        n = getattr(m,"model",None) or getattr(m,"name",None)
        if n is None and isinstance(m,dict): n=m.get("model") or m.get("name")
        if n: out.append(n)
    return out

def _wait_until_ready():
    announced=False
    while True:
        try:
            names=_model_names(client.list())
            if any(AGENT_MODEL in n for n in names):
                print(f"ollama ok; {AGENT_MODEL} is loaded"); return
            reason=f"{AGENT_MODEL} not loaded yet"
        except Exception as e:
            reason=f"server unreachable; {type(e).__name__}: {e}"
        if not announced:
            print(f"waiting for ollama, {reason}; rechecking every 10s and wont stop"); announced=True
        time.sleep(10)

def llm(prompt, cfg, counter, num_predict=None, temperature=0, think=None):
    use_think = cfg.thinking if think is None else think
    opts={"temperature":temperature, "num_predict": num_predict or cfg.decision_cap}
    attempt=0; pass_think=True
    while True:
        kw=dict(model=AGENT_MODEL, messages=[{"role":"user","content":prompt}],
                options=opts, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=use_think
        try:
            r=client.chat(**kw)
            counter.calls+=1
            try: counter.in_tok  += int(r["prompt_eval_count"] or 0)
            except Exception: pass
            try: counter.out_tok += int(r["eval_count"] or 0)
            except Exception: pass
            txt=(r["message"]["content"] or "").strip()
            if not txt:
                try: txt=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return txt
        except TypeError:
            pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60, 5*2**min(attempt-1,4)))

def embed(text, counter):
    try:
        r=client.embeddings(model=EMBED_MODEL, prompt=text or " ")
        return r["embedding"]
    except Exception:
        return None

_wait_until_ready()
print("llm and embed helpers ready")

# a plain judge/explain caller for the benchmark phase (separate from the agent's llm(); counts nothing)
def ask(prompt, num_predict=400, think=False):
    attempt=0; pass_think=True
    while True:
        kw=dict(model=JUDGE_MODEL, messages=[{"role":"user","content":prompt}],
                options={"temperature":0,"num_predict":num_predict}, keep_alive=KEEP_ALIVE)
        if pass_think: kw["think"]=think
        try:
            r=client.chat(**kw)
            t=(r["message"]["content"] or "").strip()
            if not t:
                try: t=(r["message"]["thinking"] or "").strip()
                except Exception: pass
            return t
        except TypeError: pass_think=False
        except Exception as e:
            attempt+=1
            if attempt==1 or attempt%5==0: print(f"[waiting for ollama] {type(e).__name__}: {e}; retrying")
            time.sleep(min(60,5*2**min(attempt-1,4)))
print("judge caller ready")


ollama ok; gpt-oss:120b is loaded
llm and embed helpers ready
judge caller ready


## TreeRAG engine (from newest_prototype_3)

In [4]:
import re, math, hashlib
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime
from typing import List, Dict, Any, Optional
from collections import Counter
import numpy as np


@dataclass
class TreeNode:
    node_id:str; node_type:str; name:str; path:str; summary:str
    content:str=""; children:List["TreeNode"]=field(default_factory=list)
    metadata:Dict[str,Any]=field(default_factory=dict)

    @classmethod
    def from_dict(cls,d):
        n=cls(node_id=d["node_id"],node_type=d["node_type"],name=d["name"],
              path=d.get("path",""),summary=d.get("summary",""),
              content=d.get("content",""),metadata=d.get("metadata",{}))
        n.children=[cls.from_dict(c) for c in d.get("children",[])]
        return n

    def is_leaf(self): return self.node_type=="chunk"
    def count_leaves(self): return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


@dataclass
class Question:
    qid:str; stem:str; options:Dict[str,str]; answer:str; difficulty:str=""
    answers:List[str]=field(default_factory=list)   # one or more correct letters; answer stays as the first

def _ans_list(q):
    return list(q.answers) if getattr(q,"answers",None) else ([q.answer] if q.answer else [])

def _opts_text(q):
    return "\n".join(f"{L}. {q.options[L]}" for L in "ABCDEF" if L in q.options)

def _gold_text(q):
    al=_ans_list(q)
    return "\n".join(f"{L}. {q.options.get(L,'')}" for L in al) if al else ""


@dataclass(frozen=True)
class Config:
    strategy:str="cluster"          # none chunk or cluster; how wide nodes get grouped
    working_memory:bool=True        # keep useful notes seen en route, memwalker style
    backtracking:bool=True          # allow going back up a dead end branch
    thinking:bool=False             # gpt-oss reasoning; usually the big latency lever
    group_summary:str="heuristic"   # describe virtual groups cheaply or with an llm call
    max_branch:int=6                # options per decision; fewer is easier but deeper
    decision_cap:int=512            # token cap on each nav decision; they are short
    answer_cap:int=8                # token cap on the final letter
    nav_includes_options:bool=False # new idea, show the abcd choices during navigation too
    breadcrumb:bool=False           # new idea, feed the path summaries into the final answer
    vote_samples:int=1              # new idea, self consistency; sample the answer n times and vote
    max_steps:int=24                # safety budget per question


# what makes this config differ from the plain default; used as a readable label
def config_name(cfg):
    base=Config()
    diffs=[f"{k}={getattr(cfg,k)}" for k in cfg.__dataclass_fields__ if getattr(cfg,k)!=getattr(base,k)]
    return "baseline" if not diffs else ", ".join(diffs)

def config_key(cfg):
    return hashlib.md5(repr(asdict(cfg)).encode()).hexdigest()[:12]


class Counters:
    def __init__(self): self.in_tok=0; self.out_tok=0; self.calls=0


def clip(t,n):
    t=re.sub(r"\s+"," ",t or "").strip()
    return t if len(t)<=n else t[:n]+" …"

def full(t):                              # normalise whitespace but NEVER truncate
    t=(t or "").replace("\r\n","\n")
    t=re.sub(r"[ \t]+"," ",t)
    t=re.sub(r"\n{3,}","\n\n",t)
    return t.strip()


# ---- virtual subfolders; only place embeddings are touched and only to organise groups ----
_nav_cache={}; _embed_cache={}; _gsum_cache={}

def reset_vcaches():
    _nav_cache.clear(); _gsum_cache.clear()   # embeddings persist; they dont depend on the config

def _embed_node(node,counter):
    if node.node_id in _embed_cache: return _embed_cache[node.node_id]
    v=embed((node.name+". "+(node.summary or ""))[:2000],counter)
    _embed_cache[node.node_id]=v; return v

# tiny kmeans on unit vectors so its cosine ish
def _kmeans(vectors,k,iters=25,seed=0):
    X=np.asarray(vectors,dtype=float); n=len(X); k=max(1,min(k,n))
    X=X/np.clip(np.linalg.norm(X,axis=1,keepdims=True),1e-9,None)
    rng=np.random.default_rng(seed); C=X[rng.choice(n,size=k,replace=False)].copy()
    labels=np.full(n,-1)
    for _ in range(iters):
        d=((X[:,None,:]-C[None,:,:])**2).sum(-1); new=d.argmin(1)
        if np.array_equal(new,labels): break
        labels=new
        for j in range(k):
            pts=X[labels==j]; C[j]=pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()

def _group_summary(members,label,cfg,counter):
    key=(label,cfg.group_summary,tuple(m.node_id for m in members))
    if key in _gsum_cache: return _gsum_cache[key]
    if cfg.group_summary=="llm":
        joined="\n".join(f"- {m.summary}" for m in members)
        prompt=("Synthesize the following document summaries into a single, cohesive 2-4 sentence paragraph that "
                "describes the actual facts and topics contained within them.\n"
                "CRITICAL INSTRUCTION: Do NOT mention 'this group', 'this summary', 'these documents', or use any "
                "meta-language. Write ONLY about the underlying subject matter.\n\n"+joined)
        s=llm(prompt,cfg,counter,num_predict=300)
    else:
        lines=[f"- {m.name}: {clip(m.summary,160)}" for m in members[:cfg.max_branch]]
        more=f"\n- …and {len(members)-cfg.max_branch} more" if len(members)>cfg.max_branch else ""
        s=f"a group of {len(members)} related items:\n"+"\n".join(lines)+more
    _gsum_cache[key]=s; return s

def _vid(pid,tag): return "v"+hashlib.md5(f"{pid}|{tag}".encode()).hexdigest()[:11]

def _make_vgroup(parent,members,idx,cfg,counter):
    return TreeNode(node_id=_vid(parent.node_id,f"{cfg.strategy}:{cfg.max_branch}:{idx}"),
                    node_type="vgroup",name=f"[group {idx+1} · {len(members)} items]",path="",
                    summary=_group_summary(members,f"{parent.name} group {idx+1}",cfg,counter),
                    children=list(members),metadata={"virtual":True})

def _chunk_groups(parent,kids,cfg,counter):
    size=math.ceil(len(kids)/cfg.max_branch)              # at most max_branch groups
    groups=[kids[i:i+size] for i in range(0,len(kids),size)]
    return [_make_vgroup(parent,g,i,cfg,counter) for i,g in enumerate(groups)]

def _cluster_groups(parent,kids,cfg,counter):
    vecs=[_embed_node(k,counter) for k in kids]
    if any(v is None for v in vecs): return _chunk_groups(parent,kids,cfg,counter)  # no embeds; fall back
    X=np.asarray(vecs,dtype=float); X=X/np.clip(np.linalg.norm(X,axis=1,keepdims=True),1e-9,None)
    Xc=X-X.mean(0)
    try:                                                   # order by top principal component so similar items are adjacent
        v=np.random.default_rng(0).normal(size=X.shape[1]); v/=np.linalg.norm(v)
        for _ in range(60):
            w=Xc.T@(Xc@v); nrm=np.linalg.norm(w)
            if nrm<1e-12: break
            v=w/nrm
        order=list(np.argsort(Xc@v))
    except Exception:
        order=list(range(len(kids)))
    ordered=[kids[i] for i in order]
    k=min(cfg.max_branch,len(kids)); size=math.ceil(len(kids)/k)   # equal-size buckets, never one giant blob
    groups=[ordered[i:i+size] for i in range(0,len(kids),size)]
    return [_make_vgroup(parent,g,i,cfg,counter) for i,g in enumerate(groups)]

# the children the agent chooses among, virtualised down to max_branch
def get_nav_children(node,cfg,counter):
    key=(node.node_id,cfg.strategy,cfg.max_branch,cfg.group_summary)
    if key in _nav_cache: return _nav_cache[key]
    kids=node.children
    if cfg.strategy=="none" or len(kids)<=cfg.max_branch: res=list(kids)
    elif cfg.strategy=="chunk": res=_chunk_groups(node,kids,cfg,counter)
    elif cfg.strategy=="cluster": res=_cluster_groups(node,kids,cfg,counter)
    else: res=list(kids)
    _nav_cache[key]=res; return res


# ---- the agent ----
import json as _json

def _add_memory(mem,fact,cap=20):
    fact=clip(fact,500)
    if fact and fact.lower() not in ("","none","n/a") and fact not in mem:
        mem.append(fact); del mem[:-cap]

# pull a single action out of the model json, tolerant of junk
def _parse_decision(raw,n_opts,wm):
    s=re.sub(r"^```(?:json)?|```$","",raw.strip(),flags=re.M).strip()
    m=re.search(r"\{.*\}",s,flags=re.S)
    if m:
        try:
            d=_json.loads(m.group(0)); act=str(d.get("action","")).lower().strip()
            if act not in ("descend","answer","backtrack"): act="descend" if n_opts else "answer"
            ci=d.get("child",None)
            try: ci=int(ci)
            except (TypeError,ValueError): ci=None
            return {"action":act,"child":ci,"remember":(d.get("remember") or "").strip() if wm else ""}
        except Exception: pass
    return {"action":"descend" if n_opts else "backtrack","child":0 if n_opts else None,"remember":""}

def _ask(query,node,options,memory,cfg,counter,can_back):
    opts_txt="\n".join(f"[{i}] {o.name} — {clip(o.summary,360)}" for i,o in enumerate(options)) or "(no options here)"
    acts=(['"descend"'] if options else [])+['"answer"']+(['"backtrack"'] if can_back else [])
    mem_block=""; remember_field=""
    if cfg.working_memory:
        mem_block="WORKING MEMORY (facts gathered so far):\n"+("\n".join("- "+m for m in memory) or "(empty)")+"\n\n"
        remember_field='"remember": "<a useful fact from the CURRENT summary to keep, else empty>", '
    prompt=(
        "you are an agent navigating a tree of document summaries to answer a question. "
        "you see only summaries and move one node at a time.\n\n"
        f"QUESTION: {query}\n\n{mem_block}"
        f"CURRENT NODE: {node.name} [{node.node_type}]\nCURRENT SUMMARY: {clip(node.summary,900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"choose ONE action ({', '.join(acts)}). reply with ONLY json:\n"
        '{"reasoning":"<1 sentence>", '+remember_field+
        '"action":"descend|answer|backtrack", "child":<index or null>}\n'
        "descend into the option most likely to lead to the answer; answer if you have enough; "
        "backtrack if none of these are relevant.")
    cap=cfg.decision_cap if not cfg.thinking else max(cfg.decision_cap,1024)
    return _parse_decision(llm(prompt,cfg,counter,num_predict=cap),len(options),cfg.working_memory)

SCORE_MODE = globals().get("SCORE_MODE","judge")   # judge: free response graded 0-1 by an llm; letter: pick A-D

# a short label for the compact path string
def _short(n):
    if n.metadata.get("virtual"): return "[grp]"
    return (n.name or "?")[:22]

def _src(n): return n.metadata.get("source_file") or n.path or n.name

# run the whole traversal for one question; live draws the path on a single growing line
def run_agent(q,cfg,counter,trace=True,live=False):
    ink=(lambda s: print(s,end="",flush=True)) if live else (lambda s: None)
    nav_q=q.stem                                    # the agent only ever sees the question, never the choices
    if cfg.nav_includes_options:
        nav_q+="\noptions: "+"; ".join(f"{L}) {q.options[L]}" for L in "ABCDEF" if L in q.options)
    memory=[]; evidence=[]; seen=set(); visited={ROOT.node_id}; crumbs=[]
    trail=["root"]; backtracks=0
    stack=[{"node":ROOT,"options":get_nav_children(ROOT,cfg,counter),"tried":set()}]
    ink("  path: root")
    steps=0
    while stack and steps<cfg.max_steps:
        steps+=1; fr=stack[-1]; node=fr["node"]
        present=[(i,o) for i,o in enumerate(fr["options"]) if i not in fr["tried"] and o.node_id not in visited]
        opts=[o for _,o in present]
        can_back=cfg.backtracking and len(stack)>1
        dec=_ask(nav_q,node,opts,memory,cfg,counter,can_back)
        if cfg.working_memory and dec["remember"]: _add_memory(memory,dec["remember"])
        if cfg.breadcrumb: crumbs.append(node.summary)
        act=dec["action"]
        if act=="answer" and not evidence and opts: act="descend"   # dont answer before reading a real document
        if act=="descend" and not opts:                             # nothing left here; go back up or stop
            act="backtrack" if can_back else "answer"
        if act=="answer" or (act=="backtrack" and not can_back):    # disallowed backtrack becomes answer
            if node.node_id not in seen: evidence.append(node); seen.add(node.node_id)
            ink(" ⇒ answer"); break
        if act=="backtrack":
            popped=stack.pop(); parent=stack[-1]; backtracks+=1; trail.append("↩"); ink(" ↩")
            for i,o in enumerate(parent["options"]):
                if o.node_id==popped["node"].node_id: parent["tried"].add(i); break
            continue
        ci=dec["child"]
        if ci is None or not (0<=ci<len(opts)): ci=0    # bad index; just take the first
        child=opts[ci]; visited.add(child.node_id)
        child_opts=get_nav_children(child,cfg,counter)
        if child.is_leaf() or not child_opts:           # reached a real leaf; grab its text
            if child.node_id not in seen: evidence.append(child); seen.add(child.node_id)
            if cfg.working_memory and child.content: _add_memory(memory,clip(child.content,600))
            trail.append(_short(child)+"*"); ink(f" → {child.name}✓")
            for i,o in enumerate(fr["options"]):
                if o.node_id==child.node_id: fr["tried"].add(i); break
            continue
        trail.append(_short(child)); ink(f" → {child.name}")
        stack.append({"node":child,"options":child_opts,"tried":set()})
    if live: print()                                    # close the path line
    if SCORE_MODE=="letter":
        response=_answer_letter(q,memory,evidence,crumbs,cfg,counter)
    else:
        response=_answer_response(q,memory,evidence,crumbs,cfg,counter)
    return {"response":response,"evidence":evidence,"steps":steps,
            "backtracks":backtracks,"path":" › ".join(trail)}

def _parse_letter(text):
    t=(text or "").strip().upper()
    if not t: return ""
    cues=re.findall(r"ANSWER[^A-F]{0,8}?\b([A-F])\b",t)   # answer is B, correct answer: C; take the last
    if cues: return cues[-1]
    m=re.search(r"\b([ABCDEF])\b",t) or re.search(r"([ABCDEF])",t)
    return m.group(1) if m else ""

# build the final multiple choice prompt from whatever was gathered, then vote if asked
def _answer_letter(q,memory,evidence,crumbs,cfg,counter):
    opts=_opts_text(q)
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    if evidence:
        ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1200)}"
                       for e in evidence[:MAX_EVIDENCE])
        parts.append("evidence:\n"+ev)
    ctx="\n\n".join(parts) or "(no context gathered)"
    prompt=(f"answer the multiple choice question using the gathered information.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{opts}\n\n{ctx}\n\n"
            "respond with ONLY the letter(s) of the correct option(s).")
    cap=1024 if cfg.thinking else 64
    if cfg.vote_samples<=1:
        return _parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0))
    votes=[_parse_letter(llm(prompt,cfg,counter,num_predict=cap,temperature=0.7)) for _ in range(cfg.vote_samples)]
    votes=[v for v in votes if v]
    return Counter(votes).most_common(1)[0][0] if votes else ""

# free text answer from whatever the traversal gathered; this is what the judge grades
def _answer_response(q,memory,evidence,crumbs,cfg,counter):
    parts=[]
    if cfg.working_memory and memory: parts.append("notes you gathered:\n"+"\n".join("- "+m for m in memory))
    if cfg.breadcrumb and crumbs: parts.append("summaries along your path:\n"+"\n".join("- "+clip(c,200) for c in crumbs[-8:]))
    srcs=[]
    if evidence:
        groups={}                                            # consolidate evidence under its source document
        for e in evidence[:MAX_EVIDENCE]:
            k=e.metadata.get('source_file') or e.path or e.name
            if k not in groups: groups[k]=[]; srcs.append(k)
            groups[k].append(clip(e.content or e.summary,1200))
        blocks=[f"SOURCE [{k}]:\n"+"\n".join(groups[k]) for k in srcs]
        parts.append("evidence grouped by source document:\n\n"+"\n\n".join(blocks))
    ctx="\n\n".join(parts) or "(no context gathered)"
    multi=len(srcs)>1
    cite=("this evidence spans MULTIPLE documents. attribute each claim to the specific document it came from with a "
          "bracket cite like [source_file] at the point you use it, and make sure every document you drew on is cited."
          if multi else "cite the source file in brackets where its information is used.")
    prompt=(f"answer the question using only the gathered information; be specific.\n\n"
            f"QUESTION: {q.stem}\n\n{ctx}\n\n"
            f"give the answer in 1-4 sentences. {cite}")
    return llm(prompt,cfg,counter,num_predict=1024 if cfg.thinking else 440,temperature=0)

_JUDGE_CFG=Config()     # judge runs with reasoning off

# index the tree so each chunk knows its parent section/document, for bottom-up merging
_NODES={}; _PARENT={}; MERGE_FRAC=0.6
def index_tree(root):
    _NODES.clear(); _PARENT.clear(); st=[(root,None)]
    while st:
        n,p=st.pop(); _NODES[n.node_id]=n
        if p is not None: _PARENT[n.node_id]=p
        for c in n.children: st.append((c,n.node_id))
    return _NODES,_PARENT

def all_chunks(node):
    out=[]; st=[node]
    while st:
        n=st.pop()
        if n.is_leaf(): out.append(n)
        else: st.extend(reversed(n.children))
    return out

def same_document(node):                                    # true only when every chunk under node is one document
    srcs={c.metadata.get("source_file") for c in all_chunks(node)}
    return len(srcs)==1 and None not in srcs

# bottom-up: where most of a section's chunks were kept, replace them with the whole section as one coherent unit
def merge_bottom_up(evidence):
    bypar={}; singles=[]; info=[]
    for e in evidence:
        pid=_PARENT.get(e.node_id)
        if pid is None or pid not in _NODES: singles.append(e); continue
        bypar.setdefault(pid,[]).append(e)
    out=[]
    for pid,kept in bypar.items():
        parent=_NODES[pid]; chunks=[c for c in parent.children if c.is_leaf()]
        if len(chunks)>1 and len(kept)>=math.ceil(MERGE_FRAC*len(chunks)):
            docfile=next((k.metadata.get('source_file') for k in kept if k.metadata.get('source_file')),None) or parent.path or parent.name
            whole=full("\n\n".join((c.content or c.summary or "") for c in chunks))
            md=dict(parent.metadata); md["source_file"]=docfile
            out.append(replace(parent, content=whole, metadata=md)); info.append((parent.name,len(kept),len(chunks)))
        else:
            out.extend(kept)
    out.extend(singles)
    return out, info

# final curation: the model keeps only the pieces it will actually use in the answer, however many that is
def _truthy(v):
    if isinstance(v,bool): return v
    if isinstance(v,(int,float)): return v>=0.5
    if isinstance(v,str): return v.strip().lower() in ("true","yes","y","1")
    return False

def _parse_one_ans(raw):
    m=re.search(r"\{.*\}", raw or "", re.S)
    if m:
        try:
            o=_json.loads(m.group(0)); return _truthy(o.get("answers")), str(o.get("why",""))[:200]
        except Exception: pass
    b=re.search(r'"?answers"?\s*[:=]\s*"?(true|false|yes|no)', raw or "", re.I)
    return ((b.group(1).lower() in ("true","yes")) if b else False), ""

def _parse_batch_ans(raw):
    out={}; m=re.search(r"\[.*\]", raw or "", re.S)
    if not m: return out
    try: arr=_json.loads(m.group(0))
    except Exception: return out
    for o in arr:
        try: out[int(o.get("i"))]=(_truthy(o.get("answers")), str(o.get("why",""))[:200])
        except Exception: pass
    return out

def curate(q, evidence, counter):
    # CURATION = an explicit ANSWER-relevance judgment, made PER PASSAGE. it is NOT a navigation-score
    # cutoff: the score told us where to look; "does this answer the question" is a different question,
    # and answering it is what curation decides. each passage is judged on its own, so a real hit can
    # never be lost to a free-form "pick a subset" omission (which is what dropped the 0.95 before).
    if not evidence: return [], []
    ask=llm("State, in one precise sentence, exactly what a correct answer to this question must provide "
            "(the specific thing being asked for; name the actor/situation if the question does).\n\n"
            f"QUESTION: {q.stem}\n\nReply with ONLY that one sentence.",
            _JUDGE_CFG,counter,num_predict=80,temperature=0,think=False).strip()
    print(f"  \u2316 curation target: {ask}")

    RUBRIC=("Include a passage ONLY if it DIRECTLY and EXPLICITLY states part of the answer \u2014 its OWN "
            "words answer what is asked. EXCLUDE passages that are merely on-topic, or that give background, "
            "scope, purpose, or definitions, or that describe a DIFFERENT actor, step, or situation than the "
            "one the question asks about (e.g. another role's duties when the question asks about THIS role).")

    def _ask_one(e):
        raw=llm("Decide whether ONE passage should be included as evidence for the final answer. "+RUBRIC+"\n\n"
                f"QUESTION: {q.stem}\nA CORRECT ANSWER MUST PROVIDE: {ask}\n\n"
                f"PASSAGE ({e.path or e.name}):\n{full(e.content or e.summary)}\n\n"
                'reply ONLY json {"answers": true or false, "why": "<one line: what part it answers, or why not>"}',
                _JUDGE_CFG,counter,num_predict=140,temperature=0,think=False)
        return _parse_one_ans(raw)

    items=list(enumerate(evidence)); B=6; verdict={}
    for st in range(0,len(items),B):
        chunk=items[st:st+B]   # [(global_idx, node), ...]
        listing="\n\n".join(f"[{li}] ({e.path or e.name})\n{full(e.content or e.summary)}"
                              for li,(gi,e) in enumerate(chunk))
        raw=llm("For EACH passage below, decide whether it should be included as evidence for the final "
                "answer. "+RUBRIC+"\n\n"
                f"QUESTION: {q.stem}\nA CORRECT ANSWER MUST PROVIDE: {ask}\n\nPASSAGES:\n{listing}\n\n"
                'reply ONLY a json list with ONE object per passage, using the [bracketed] index: '
                '[{"i":<idx>,"answers":true or false,"why":"<one line>"}].',
                _JUDGE_CFG,counter,num_predict=60*len(chunk)+120,temperature=0,think=False)
        parsed=_parse_batch_ans(raw)
        for li,(gi,e) in enumerate(chunk):
            verdict[gi]=parsed[li] if li in parsed else _ask_one(e)   # individual fallback if model skipped one

    kept=[]
    for gi,e in items:
        ans,why=verdict.get(gi,(False,""))
        e.metadata['_ans']=bool(ans); e.metadata['_why']=why
        if ans: kept.append(e)
    if not kept:                                  # never empty: keep the single best navigation hit, flagged
        best=max(evidence,key=lambda e:float(e.metadata.get('_rel',0.0)))
        best.metadata['_ans']=True
        best.metadata['_why']="fallback: no passage was judged a direct answer; kept top navigation hit"
        kept=[best]
    keep_ids={id(e) for e in kept}
    keep=[i for i,e in enumerate(evidence) if id(e) in keep_ids]
    return kept, keep

# ---- OICR retrieval-agent prompts and the navigation decisions used by the greedy search ----
DOMAIN="Laboratory Quality Management"
ORGANIZATION="Ontario Institute for Cancer Research (OICR)"
DUNNO="The requested information could not be found."
SYSTEM_PROMPT=("You are a skilled information retrieval agent for "
  f"a {DOMAIN} document library for the {ORGANIZATION} genomics laboratory.\n"
  "Your task is to research answers to user questions using the provided documents.\n"
  f"Today's date is {datetime.today().strftime('%Y-%m-%d')}. "
  "'We' or 'us' typically refers to OICR's genomics team.")
INSTRUCTIONS=("Answer user questions ONLY using the information in the "
  "files. Do not use any prior knowledge in your answers. "
  "Do not speculate. If the answer cannot be found in the "
  f"files, say '{DUNNO}'\n"
  "If possible, return only exact, direct quotes of the most "
  "relevant passages from the documents followed by a "
  "reference to the respective document ID. Format your "
  "reference as [doc_id], where `doc_id` is the document ID.\n"
  "If a question "
  "requires a more detailed answer which cannot be expressed "
  "in less than one hundred words, refer the user to the "
  "relevant primary source documents instead, citing their "
  "document IDs following the above formatting guideline. "
  "For example, if a user asks for a specific SOP, simply "
  "provide the name of the SOP document and cite its "
  "document ID.\n")

def all_chunks(node):
    out=[]; st=[node]
    while st:
        n=st.pop()
        if n.is_leaf(): out.append(n)
        else: st.extend(reversed(n.children))
    return out

# whole document/section as one evidence item carrying its full text
def whole_unit(node):
    chunks=all_chunks(node)
    whole=full("\n\n".join((c.content or c.summary or "") for c in chunks))
    src=next((c.metadata.get('source_file') for c in chunks if c.metadata.get('source_file')),None) or node.path or node.name
    md=dict(node.metadata); md["source_file"]=src
    return replace(node, content=whole, metadata=md)

# like whole_unit, but first drop paragraphs that are GENUINELY unrelated to the question.
# keep-biased: a paragraph is removed only if it could not be used in ANY part of the answer;
# anything that might contribute is kept, and coherent lists/procedures are kept whole.
def prune_unit(node, query, counter):
    chunks=all_chunks(node)                                  # paragraph-level units under the section
    src=next((c.metadata.get('source_file') for c in chunks if c.metadata.get('source_file')),None) or node.path or node.name
    def _tag(c):
        md=dict(c.metadata); md.setdefault("source_file",src); return replace(c, metadata=md)
    if len(chunks)<=1:
        return [_tag(c) for c in chunks], 0, len(chunks)
    listing="\n\n".join(f"[{i}] {full(c.content or c.summary)}" for i,c in enumerate(chunks))
    raw=llm("you are about to answer the QUESTION using only the paragraphs below, taken from one section "
            "that is relevant to it. select the paragraphs whose information would actually be USED IN THE "
            "FINAL ANSWER \u2014 the facts you would state or directly rely on. drop a paragraph if it would "
            "not make it into the answer, even when it is loosely on-topic (background, scope, purpose, "
            "version history, boilerplate). do NOT drop a paragraph you would need to state or support the "
            "answer; if a paragraph belongs to a continuous list or procedure the answer uses, keep that whole "
            "list. when genuinely unsure whether a paragraph is needed, keep it.\n\n"
            f"QUESTION: {query}\n\nPARAGRAPHS:\n{listing}\n\n"
            'reply ONLY json {"keep": [indices of paragraphs that belong in the answer]}',
            _JUDGE_CFG,counter,num_predict=300,temperature=0,think=False)
    keep=None
    m=re.search(r"\{.*\}",raw or "",re.S)
    if m:
        try:
            arr=_json.loads(m.group(0)).get("keep",[]); keep=[int(i) for i in arr if 0<=int(i)<len(chunks)]
        except Exception: keep=None
    if not keep: keep=list(range(len(chunks)))               # parse failed/empty -> keep all, never lose context
    kept=[_tag(chunks[i]) for i in sorted(set(keep))]
    return kept, len(chunks)-len(kept), len(chunks)

# classify the query: lookup -> the answer lives in ONE file; synthesis -> it must be assembled across several files
def classify_query(q, counter):
    raw=llm("classify the question by HOW its answer must be assembled.\n"
            "- lookup    : the answer is a specific fact, value, definition, or the contents of ONE thing, "
            "found within a SINGLE document \u2014 even if that document holds a list. typical forms: "
            "'what is X', 'what is part of X', 'define X', 'which SOP/document contains X', 'what does X require'.\n"
            "- synthesis : answering well requires ENUMERATING or AGGREGATING across MANY documents/records \u2014 "
            "no single document holds the whole answer. typical forms: 'what are OUR X', 'which X do we have', "
            "'list all X', 'how many X', anything scoped to an inventory spread across the corpus.\n\n"
            "EXAMPLES:\n"
            "Q: What is part of an assay validation? -> lookup   (the components are defined together in one SOP)\n"
            "Q: What are our validated clinical assays? -> synthesis   (must be gathered across many documents)\n"
            "Q: Which SOP describes reagent qualification? -> lookup\n"
            f"QUESTION: {q.stem}\n\n"
            'reply ONLY json {"type":"lookup" or "synthesis"}',
            _JUDGE_CFG,counter,num_predict=60,temperature=0,think=False)
    m=re.search(r'"?type"?\s*[:=]\s*"?(lookup|synthesis)"?', raw or "", re.I)
    return m.group(1).lower() if m else "lookup"

# at a document/section node, judge its OWN summary: if everything it describes is relevant, take the whole unit
def summary_gate(query, node, counter):
    raw=llm("you are deciding whether to use a whole document section as-is, or to look deeper inside it. read its "
            "summary. if EVERY part of what the summary describes is relevant to the question (it is wholly on-topic, "
            "for example it comprehensively covers exactly what is asked), we take the whole thing. if the summary "
            "also covers aspects unrelated to the question, we should look deeper at its parts.\n\n"
            f"QUESTION: {query}\nSECTION: {node.name}\nSUMMARY: {clip(node.summary,900)}\n\n"
            'reply ONLY json {"whole": true or false, "reason": "<one line>"}',
            _JUDGE_CFG,counter,num_predict=200,temperature=0,think=False)
    m=re.search(r"\{.*\}",raw or "",re.S)
    if m:
        try:
            o=_json.loads(m.group(0)); return bool(o.get("whole")), str(o.get("reason",""))[:160]
        except Exception: pass
    b=re.search(r'"?whole"?\s*[:=]\s*(true|false|yes|no)', raw or "", re.I)   # tolerate fences/prose/non-json
    if b: return b.group(1).lower() in ("true","yes"), "loose-parsed"
    
    # Aggressive fallback if the model just blurts out "true" in a sentence
    if "true" in (raw or "").lower(): return True, "loose-parsed (fallback true)"
    return False, "gate parse failed; descend"

# is this single excerpt relevant enough to keep? cheap per-piece relevance gate
def leaf_decision(query, node, qtype, counter):
    raw=llm("Judge whether this document excerpt should be kept to help answer the question. "
            "CRITICAL: Keep it if it contains specific facts, procedural steps, or rules that directly contribute to the answer. "
            "Reject it ONLY if it is purely incidental, vague background fluff, or completely unrelated. "
            "When in doubt, or if it provides a needed piece of a larger procedure, set 'relevant' to true.\n\n"
            f"QUESTION: {query}\n\nEXCERPT ({node.metadata.get('source_file') or node.name}):\n"
            f"{clip(node.content or node.summary,1500)}\n\n"
            'reply ONLY json {"relevant": true or false}',
            _JUDGE_CFG,counter,num_predict=80,temperature=0,think=False)
    m=re.search(r'"?relevant"?\s*[:=]\s*"?(true|false|yes|no)', raw or "", re.I)
    keep = (m.group(1).lower() in ("true","yes")) if m else False   # parse miss -> drop, keep evidence lean
    return keep, False
    
# completeness halt: do the pieces gathered SO FAR, taken together, fully answer the question? if yes, stop searching
# completeness halt: triggers the moment AT LEAST ONE relevant fact is found
# completeness halt: triggers extremely easily to prevent over-searching
# completeness halt: balanced to find a sufficient answer without over-searching
# completeness halt: do the pieces gathered SO FAR, taken together, fully answer the question?
def is_complete(query, evidence, counter):
    if not evidence: return False
    ordered = list(reversed(evidence))[:MAX_EVIDENCE]   # newest (just-committed) unit first
    blocks=[f"[{e.metadata.get('source_file') or e.path or e.name}]\n{full(e.content or e.summary)}"
            for e in ordered]
    prompt=(
        "Decide whether the gathered evidence DIRECTLY answers the question.\n\n"
        "1. Copy the exact sentence(s) from the evidence that STATE the answer, verbatim. "
        "If no sentence states it and you would have to infer, assume, or stitch together distant "
        "hints, the quote is NONE.\n"
        "2. Set directly_stated=true ONLY if those quote(s) answer the question on their own with no "
        "inference. If the evidence merely implies, relates to, or partially touches the answer, false.\n\n"
        "Calibration:\n"
        "- You do NOT need exhaustiveness or confirmation from other documents. If it is explicitly "
        "stated here, that is enough — stop.\n"
        "- 'I could guess a reasonable answer from this' is NOT enough; the text must actually say it.\n"
        "- If the question asks for several specific items, it is direct only if those items are "
        "explicitly present in the quotes.\n\n"
        f"QUESTION: {query}\n\nGATHERED EVIDENCE:\n"+"\n\n".join(blocks)+
        '\n\nreply ONLY json {"quote": "<verbatim supporting text, or NONE>", "directly_stated": true or false}'
    )
    raw=llm(prompt,_JUDGE_CFG,counter,num_predict=500,temperature=0,think=False)
    if re.search(r'"quote"\s*:\s*"?\s*none\b', raw or "", re.I): return False   # no quotable span → not done
    d=re.search(r'"?directly_stated"?\s*[:=]\s*(true|false|yes|no)', raw or "", re.I)
    return bool(d) and d.group(1).lower() in ("true","yes")


def answer_oicr(q, evidence, counter):
    groups={}; order=[]
    for e in evidence[:MAX_EVIDENCE]:
        k=e.metadata.get('source_file') or e.path or e.name
        if k not in groups: groups[k]=[]; order.append(k)
        groups[k].append(full(e.content or e.summary))

    docs="\n\n".join(f"--- START DOCUMENT [doc_id: {k}] ---\n"+"\n".join(groups[k])+"\n--- END DOCUMENT ---" for k in order) or "(no documents were retrieved)"

    prompt=(
        f"{SYSTEM_PROMPT}\n\n{INSTRUCTIONS}\n\n"
        "Write your reply in three parts under these exact headings:\n\n"
        "THOUGHTS:\n<your reasoning and the exact passages you are relying on>\n\n"
        "FINAL_ANSWER:\n<the answer itself>\n\n"
        "SUPPORTING_QUOTES:\n<each verbatim passage you relied on, one per line, each ending in [doc_id]>\n\n"
        "Rules for FINAL_ANSWER:\n"
        "- Answer the question DIRECTLY with the substance. Do NOT restate, rephrase, or echo the question, and do "
        "not open with a stem such as 'Parts of ... include' or 'X is defined as'. Begin with the actual content.\n"
        "- State the real facts, not a description of where they live. Cite the source as [doc_id] after each claim.\n"
        f"- Use ONLY the documents above. If the answer is not in them, write exactly: '{DUNNO}'\n"
        "- Be concise and self-contained. Keep to 100 words. \n\n"
        f"DOCUMENTS:\n{docs}\n\n"
        f"QUESTION: {q.stem}\n\n"
        "THOUGHTS:\n"
    )

    # FIX: Bumped num_predict from 800 to 2048 to give it plenty of room to think and answer.
    # Alternatively, you can use num_predict=-1 to rely entirely on the model's context window.
    raw = llm(prompt,_JUDGE_CFG,counter,num_predict=2048,temperature=0,think=False)

    clean_raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)
    # split off the quotes section first, then isolate the answer
    aq = re.split(r'SUPPORTING_QUOTES:', clean_raw, flags=re.I)
    quotes = aq[1].strip() if len(aq) > 1 else ""
    parts = re.split(r'FINAL_ANSWER:', aq[0], flags=re.I)
    ans = parts[-1] if len(parts) > 1 else parts[0]
    ans = re.sub(r'^.*?(?:answer could be:|concise answer is:|the source says:|\[user request\]:)\s*', '', ans, flags=re.I|re.S)
    ans = ans.strip(' "\'\n`-:•–')

    # keep only quotes that actually appear in the gathered evidence (drops paraphrases/inventions)
    ev_text = " ".join(full(e.content or e.summary) for e in evidence).lower()
    real = [q.strip() for q in quotes.splitlines()
            if q.strip() and re.sub(r'\[.*?\]', '', q).strip()[:40].lower() in ev_text]
    quotes = "\n".join(real)

    return ans, quotes

# checks if a section's summary describes a unified concept (like a list or procedure)
# checks if a section's sub-nodes describe a unified concept (like a list or procedure)
def check_coherence(query, node, kids, counter):
    # Grab the names of the children to show the LLM the structure
    kid_names = "\n".join([f"- {k.name}" for k in kids[:20]]) 
    
    prompt=(
        "You are evaluating a document section to decide if its contents should be kept together.\n\n"
        f"QUESTION: {query}\n"
        f"SECTION NAME: {node.name}\n"
        f"SUB-SECTIONS:\n{kid_names}\n\n"
        "Do these sub-sections represent a single coherent unit (like a continuous numbered list, a step-by-step procedure, or tightly connected paragraphs)?\n"
        "CRITICAL: If the sub-sections are sequentially numbered (e.g., 1., 2., 3.) or represent consecutive steps of a single process, they ARE coherent and must not be separated.\n"
        "Reply 'COHERENT' if they belong together and should be extracted as a whole.\n"
        "Reply 'SEPARATE' if they are disconnected topics and we should be picky.\n\n"
        "OUTPUT:"
    )
    raw=llm(prompt,_JUDGE_CFG,counter,num_predict=20,temperature=0,think=False)
    return "coherent" in (raw or "").lower()

def _parse_judge(raw):
    score = 0.0
    reason = ""
    
    # 1. Strip <think> tags (handles cases where the model forces reasoning)
    clean_raw = re.sub(r'<think>.*?</think>', '', raw or "", flags=re.S).strip()
    
    # 2. Strip markdown code blocks just in case
    clean_raw = re.sub(r"^```(?:json)?|```$","", clean_raw, flags=re.M).strip()
    
    print(f"2. CLEANED TEXT (post-regex stripping):\n{repr(clean_raw)}\n")
    
    # 3. Find and parse the JSON block
    m = re.search(r"\{.*\}", clean_raw, flags=re.S)
    
    parsed_successfully = False

    if m:
        json_str = m.group(0)
        
        # Attempt 1: Strict JSON
        try:
            d = _json.loads(json_str)
            score = float(d.get("score", 0))
            reason = str(d.get("reason", "")).strip()
            parsed_successfully = True
        except Exception:
            # Attempt 2: AST Eval (Handles single quotes, trailing commas)
            try:
                import ast
                d = ast.literal_eval(json_str)
                score = float(d.get("score", 0))
                reason = str(d.get("reason", "")).strip()
                parsed_successfully = True
            except Exception:
                pass

    # 4. Fallback: Plain Text Regex Extraction
    if not parsed_successfully:
        # Look for "Score: 0.8" or '"score": 1'
        score_match = re.search(r'(?:score)[\s"\'=:]+([01](?:\.\d+)?|0?\.\d+)', clean_raw, re.I)
        if score_match:
            score = float(score_match.group(1))
        else:
            # Last resort bare number: "0.8", "1.0", "1", "0"
            bare_match = re.search(r'\b(0\.\d+|1\.0+|0|1)\b', clean_raw)
            if bare_match:
                score = float(bare_match.group(1))

        # Look for "Reason: It matches..." or '"reason": "It matches..."'
        reason_match = re.search(r'(?:reason)[\s"\'=:]+(.*)', clean_raw, re.I)
        if reason_match:
            # Strip trailing quotes, braces, or whitespace
            reason = reason_match.group(1).strip(' "\'}')
        else:
            # If no reason label is found, just grab the first chunk of text as the reason
            reason = clean_raw[:150].replace('\n', ' ').strip()

    final_score = max(0.0, min(1.0, score))
            
    return final_score, reason

# grade the agents free response against the correct option(s) by MEANING; returns score and a one line reason
def judge_score(q,response,counter):
    al=_ans_list(q); multi=len(al)>1
    prompt=("you are a fair grader. a student answered an open question in their own words and could not see the "
            "choices. the multiple choice version below has the correct option(s) marked and those are the ground "
            "truth. the correct answer may be ONE OR MORE options.\n\n"
            f"QUESTION: {q.stem}\nOPTIONS:\n{_opts_text(q)}\nCORRECT OPTION(S): {', '.join(al)}\n{_gold_text(q)}\n\n"
            f"STUDENT RESPONSE:\n{response or '(empty)'}\n\n"
            "grade from 0.0 to 1.0 how well the response matches the MEANING of the correct option(s). judge by "
            "meaning not wording. "
            +("when several options are correct, give full credit only if the response conveys ALL of them, and "
              "proportional partial credit for covering some. " if multi else
              "give full or near full credit when it conveys the correct idea even in different words, partial when "
              "incomplete. ")
            +"give low credit when it matches a wrong option or is irrelevant. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence comparing the response to the correct option(s)>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=1024,temperature=0,think=False))

# grade the agents free response against the correct option(s) by MEANING; returns score and a one line reason
def judge_score_new(q,response,counter):
    al=_ans_list(q); multi=len(al)>1
    prompt=("you are a fair grader. a student answered an open question in their own words and could not see "
            "the choices. the multiple choice version below has the correct option(s) marked; those are the "
            "ground truth.\n\n"
            "GRADE ONLY ON THIS:\n"
            "- Does the answer convey the substance of the correct option(s)? Full credit if yes.\n"
            "- Multiple correct options: credit is proportional to how many are covered in meaning.\n\n"
            "DO NOT PENALIZE:\n"
            "- extra information, context, detail, or correct facts beyond what was asked — as long as it does "
            "not assert a wrong option as true. additional correct or neutral content is fine and must not "
            "lower the score.\n"
            "- verbosity, phrasing, structure, or the answer not naming the option explicitly.\n\n"
            "ONLY PENALIZE WHEN:\n"
            "- a correct option's substance is missing, OR\n"
            "- the answer states/endorses an incorrect option as true, OR directly contradicts a correct one.\n"
            "mere mention of a wrong topic is NOT endorsement; it only counts against the answer if asserted as "
            "the correct answer.\n\n"
            f"QUESTION: {q.stem}\n\nOPTIONS:\n{opts}\n\nSTUDENT ANSWER:\n{response}\n\n"
            'reply ONLY json {"score": <0.0-1.0>, "reason": "<one line>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=1024,temperature=0,think=False))

# grade whether the RETRIEVED text contains the facts needed for the correct answer(s); isolates navigation from answering
def judge_evidence(q,evidence,counter):
    ev="\n\n".join(f"[{e.metadata.get('source_file') or e.path or e.name}] {clip(e.content or e.summary,1500)}"
                   for e in evidence[:MAX_EVIDENCE]) or "(nothing was retrieved)"
    prompt=("you are checking whether a retrieval system fetched the right information, not whether anyone answered. "
            "below is the correct answer(s) to a question and the text the system retrieved.\n\n"
            f"QUESTION: {q.stem}\nCORRECT ANSWER(S): {', '.join(_ans_list(q))}\n{_gold_text(q)}\n\n"
            f"RETRIEVED TEXT:\n{ev}\n\n"
            "rate from 0.0 to 1.0 how well the retrieved text CONTAINS the information needed to reach the correct "
            "answer(s), whether or not it is phrased as the answer; if several answers are correct, weight by how "
            "many are supported. 1.0 means the needed facts are clearly present, 0.0 means absent. reply with ONLY json:\n"
            '{"score": <number 0.0 to 1.0>, "reason": "<one concise sentence>"}')
    return _parse_judge(llm(prompt,_JUDGE_CFG,counter,num_predict=1024,temperature=0,think=False))

print("eval core ready; configs, virtual subfolders and the agent are defined")

# ---- running-doc answering: ONE pass. walk pieces strongest-first; a piece is folded into a working
# ---- doc ONLY if it directly answers (part of) the question; the final call just reorganises that doc.
MAX_EVAL = 60   # safety cap on how many pieces we evaluate (relevance-sorted, strongest first)

def build_running_doc(q, evidence, counter):
    pieces = sorted(evidence, key=lambda e: -float(e.metadata.get('_rel',0.0)))[:MAX_EVAL]
    notes=[]; contributions=[]
    for e in pieces:
        rel=float(e.metadata.get('_rel',0.0))
        loc=e.path or e.name
        src=e.metadata.get('source_file') or e.path or e.name
        body=full(e.content or e.summary)
        sofar="\n\n".join(f"[{s}] {m}" for s,m in notes) or "(empty)"
        raw=llm(
            "You are assembling the material to answer a QUESTION, one source piece at a time.\n"
            "Decide whether the NEW PIECE directly provides information that answers the QUESTION "
            "(in whole or in part). If YES, copy out VERBATIM only the specific sentences, items, or "
            "values from the NEW PIECE that answer it \u2014 nothing that is background, scope, purpose, "
            "or already covered by the material so far. If it does not directly answer, add nothing.\n\n"
            f"QUESTION: {q.stem}\n\nMATERIAL SO FAR:\n{sofar}\n\n"
            f"NEW PIECE [doc_id: {src}]:\n{body}\n\n"
            'reply ONLY json {"answers": true or false, "material": "<verbatim extract to add, or empty>"}',
            _JUDGE_CFG, counter, num_predict=600, temperature=0, think=False)
        ans=False; mat=""
        m=re.search(r"\{.*\}", raw or "", re.S)
        if m:
            try:
                o=_json.loads(m.group(0)); ans=bool(o.get("answers")); mat=str(o.get("material","")).strip()
            except Exception: ans=False; mat=""
        used = ans and len(mat)>=3
        if used: notes.append((src, mat))
        contributions.append({"rel":rel,"loc":loc,"src":src,"answered":used,"material":mat if used else ""})

    working_doc="\n\n".join(f"[{s}] {m}" for s,m in notes)
    if not notes:
        return DUNNO, "", working_doc, contributions

    # The running doc IS the kept set. The answer must reflect EVERY item in it, merged and reorganised
    # \u2014 never silently dropping a distinct fact (that recency-drop was losing useful items 1 & 3).
    numbered="\n".join(f"[item {i+1}] (cite as [{s}]) {m}" for i,(s,m) in enumerate(notes))

    def _assemble(extra=""):
        return llm(
            f"{SYSTEM_PROMPT}\n\n"
            "Answer user questions ONLY using the information below. Do not use any prior knowledge in your answers. "
            "Do not speculate. If the answer cannot be found in the "
            f"files, say '{DUNNO}'\n"
            "If possible, return only exact, direct quotes of the most "
            "relevant passages from the documents followed by a "
            "reference to the respective document ID. Format your "
            "reference as [doc_id], where `doc_id` is the document ID.\n"
            "If a question "
            "requires a more detailed answer which cannot be expressed "
            "in less than one hundred words, refer the user to the "
            "relevant primary source documents instead, citing their "
            "document IDs following the above formatting guideline. "
            "For example, if a user asks for a specific SOP, simply "
            "provide the name of the SOP document and cite its "
            "document ID.\n"
            "Do not drop, skip, or shorten away any distinct fact, step, name, or value.\n"
            f"{extra}"
            "Write under these exact headings:\n"
            "FINAL_ANSWER:\n<the answer, reflecting every item>\n\n"
            "SUPPORTING_QUOTES:\n<each verbatim passage you relied on, one per line, each ending in [doc_id]>\n\n"
            f"MATERIAL:\n{numbered}\n\n"
            f"QUESTION: {q.stem}\n\nFINAL_ANSWER:\n",
            _JUDGE_CFG, counter, num_predict=2048, temperature=0, think=False)

    def _parse(raw):
        clean=re.sub(r'<think>.*?</think>','',raw or "",flags=re.S)
        aq=re.split(r'SUPPORTING_QUOTES:',clean,flags=re.I)
        qz=aq[1].strip() if len(aq)>1 else ""
        parts=re.split(r'FINAL_ANSWER:',aq[0],flags=re.I)
        a=(parts[-1] if len(parts)>1 else parts[0]).strip(' "\'\n`-:\u2022\u2013')
        return a,qz

    ans_txt,quotes=_parse(_assemble())

    # coverage guard: if the draft missed any item, do ONE repair pass naming the missing items.
    chk=llm("For each NUMBERED item, decide whether its specific information is reflected in the ANSWER.\n\n"
            f"ANSWER:\n{ans_txt}\n\nITEMS:\n{numbered}\n\n"
            'reply ONLY json {"missing":[item numbers whose information is NOT reflected]}.',
            _JUDGE_CFG,counter,num_predict=120,temperature=0,think=False)
    missing=[]
    mm=re.search(r"\{.*\}",chk or "",re.S)
    if mm:
        try: missing=[int(x) for x in _json.loads(mm.group(0)).get("missing",[]) if 1<=int(x)<=len(notes)]
        except Exception: missing=[]
    if missing:
        miss="\n".join(numbered.splitlines()[i-1] for i in missing)
        a2,q2=_parse(_assemble(extra=f"A PRIOR DRAFT OMITTED these items \u2014 they MUST appear in the answer too:\n{miss}\n\n"))
        if a2: ans_txt,quotes=a2,(q2 or quotes)

    ev_text=" ".join(full(e.content or e.summary) for e in evidence).lower()
    real=[ln.strip() for ln in quotes.splitlines()
          if ln.strip() and re.sub(r'\[.*?\]','',ln).strip()[:40].lower() in ev_text]
    quotes="\n".join(real)
    return ans_txt, quotes, working_doc, contributions


eval core ready; configs, virtual subfolders and the agent are defined


In [5]:
import heapq
CFG = Config(strategy=STRATEGY, max_branch=MAX_BRANCH, group_summary=GROUP_SUMMARY,
             thinking=THINKING, max_steps=MAX_STEPS)

# rate one small batch of children; think OFF so the json stays clean, with a retry and an honest parse flag
def _score_batch(query, node, batch, counter):
    listing="\n".join(f"[{i}] {c.name}: {clip(c.summary,320)}" for i,c in enumerate(batch))
    prompt=("You are an expert navigator searching a tree of document summaries to answer a question. "
            "For each option, rate 0.0 to 1.0 how likely it contains or leads to that information.\n\n"
            "CRITICAL RULES:\n"
            "1. Map specific questions to their broader procedural 'home'. For example, questions about 'change requests' or 'revisions' almost always live in 'Document Control'.\n"
            "2. Look past superficial titles and score on conceptual relevance. A section whose title or summary sounds off-topic may still hold the answer in its body \u2014 do not score it near zero on the title alone.\n\n"
            f"QUESTION: {query}\nYOU ARE AT: {node.name}\nOPTIONS:\n{listing}\n\n"
            f'reply with ONLY a json list of {len(batch)} objects in order, like '
            '[{"i":0,"score":0.8,"reason":"..."}].')
    raw=llm(prompt,CFG,counter,num_predict=220*len(batch)+200,temperature=0,think=False)
    p=_parse_scores(raw,len(batch))
    if p is None:                                            # retry with a dead-simple format
        raw2=llm(f"QUESTION: {query}\nrate each option 0..1 for relevance.\n{listing}\n\n"
                 f"reply ONLY {len(batch)} lines, each like '0: 0.7'.",CFG,counter,
                 num_predict=20*len(batch)+80,temperature=0,think=False)
        p=_parse_simple(raw2,len(batch))
    if p is None:
        return [0.5]*len(batch), [""]*len(batch), [False]*len(batch)   # genuine failure; flag it, do not hide it
    sc,rs=p; return sc, rs, [True]*len(batch)

def _parse_scores(raw,n):
    m=re.search(r"\[.*\]",raw or "",re.S)
    if not m: return None
    try: arr=json.loads(m.group(0))
    except Exception: return None
    sc=[None]*n; rs=[""]*n
    for o in arr:
        try:
            i=int(o.get("i")); 
            if 0<=i<n: sc[i]=max(0.0,min(1.0,float(o.get("score",0.5)))); rs[i]=str(o.get("reason",""))
        except Exception: pass
    if any(s is None for s in sc): return None
    return sc,rs

def _parse_simple(raw,n):
    sc=[None]*n
    for line in (raw or "").splitlines():
        mm=re.match(r"\s*\[?(\d+)\]?\s*[:=)]\s*([01](?:\.\d+)?)",line)
        if mm:
            i=int(mm.group(1))
            if 0<=i<n: sc[i]=max(0.0,min(1.0,float(mm.group(2))))
    if any(s is None for s in sc): return None
    return sc, [""]*n

# score all children in safe-sized batches so a 30-wide folder never truncates the json
def robust_score(query, node, children, counter):
    n=len(children); B=6; scores=[0.5]*n; reasons=[""]*n; oks=[False]*n
    for st in range(0,n,B):
        sc,rs,ok=_score_batch(query,node,children[st:st+B],counter)
        for j in range(len(sc)): scores[st+j]=sc[j]; reasons[st+j]=rs[j]; oks[st+j]=ok[j]
    return scores, reasons, oks

# at each node, rate every child once; ALL relevant -> take the whole unit; SOME -> descend into those; NONE -> prune
def is_textual_unit(node): return node.node_type in ("document","section")

# extract from ONE file the passages that are actually relevant. NO whole-file dumping: we keep individual
# chunks (and recurse into sub-sections), so the evidence is precise and the answer stays coherent. if every
# passage in the file is relevant we end up keeping all of them anyway — just without the off-topic tail that
# the old "majority relevant -> take the whole thing" rule kept dragging in. appends into `evidence`.
# Extract from ONE file the passages that are actually relevant.
# This runs ONLY after the agent has committed to a document.
# Extract from ONE file the passages that are actually relevant.
# Includes an Early Stop check so it bails out the second it has a complete answer.
# Extract from ONE file the passages that are actually relevant.
# Includes an Early Stop check and a Coherence Gate for processes/lists.
DRILL_AT = 0.10      # below this, a section is too off-topic to even drill into
KEEP_LEAF_AT = 0.70  # a passage is only kept if the scorer is strongly confident (score >= this)

def _tag_rel(node, sc):                  # stamp the navigation relevance onto a kept piece
    md=dict(node.metadata); md['_rel']=float(sc); return replace(node, metadata=md)

def mine_unit(node, query, qtype, counter, evidence, depth):
    # the cheap summary score ONLY orders children and drops the clearly-dead (< DRILL_AT).
    # NOTHING is kept on score alone. the keep decision always reads real text:
    #   - passage      -> leaf_decision (a true relevance gate; rejects unrelated text)
    #   - section >=0.50 -> prune_unit (already confident it's on-topic; just trim paragraphs)
    #   - section <0.50  -> DRILL in and let leaf_decision judge each paragraph,
    #                       because a misleading summary can score the real answer low.
    # so a junk 0.15 branch can no longer sneak into evidence or burn a file slot, and the
    # answer-bearing section that the summary under-scored still gets reached. no early halt.
    kids = node.children
    if not kids:
        keep, _ = leaf_decision(query, node, qtype, counter)
        if keep: evidence.append(_tag_rel(node, KEEP_LEAF_AT))
        return bool(keep)

    scores, reasons, oks = robust_score(query, node, kids, counter)
    order = sorted(range(len(kids)), key=lambda i: -scores[i])
    print(f"{'  '*depth}  \u2261 mining {node.node_type} '{node.name}' \u2014 deep extraction mode:")
    print(f"{'  '*depth}    relevance of all {len(kids)} sub-unit(s), ranked:")
    for _i in order:
        print(f"{'  '*depth}      {scores[_i]:.2f}  [{'leaf' if kids[_i].is_leaf() else 'sect'}] {clip(kids[_i].name,60)}"
              + (f"  \u2014 {clip(reasons[_i],90)}" if reasons[_i] else ""))

    kept_any = False
    for i in order:
        child = kids[i]; s = scores[i]
        if s < DRILL_AT:
            print(f"{'  '*depth}    \u00b7 skipping {'passage' if child.is_leaf() else 'sub-section'} {clip(child.name,46)} ({s:.2f})")
            continue

        if child.is_leaf():
            if s >= KEEP_LEAF_AT:                                        # only strongly-relevant passages are worth keeping
                evidence.append(_tag_rel(child, s)); kept_any = True
                print(f"{'  '*depth}    \u2713 kept passage {child.name} ({s:.2f})")
            else:
                print(f"{'  '*depth}    \u00b7 skipping passage {child.name} ({s:.2f})")
        else:
            if s >= RELEVANT_AT:                                          # confidently on-topic: take whole section for context
                kept_units, n_drop, n_tot = prune_unit(child, query, counter)
                if kept_units:
                    evidence.extend(_tag_rel(u, s) for u in kept_units); kept_any = True
                    print(f"{'  '*depth}    \u2702 {clip(child.name,40)} ({s:.2f}): kept {len(kept_units)}/{n_tot} paragraph(s)"
                          + (f"; dropped {n_drop}" if n_drop else ""))
                else:
                    print(f"{'  '*depth}    \u00b7 {clip(child.name,40)} ({s:.2f}): nothing usable inside")
            else:                                                        # uncertain: drill, text decides
                print(f"{'  '*depth}    \u2192 into sub-section {clip(child.name,46)} ({s:.2f})")
                if mine_unit(child, query, qtype, counter, evidence, depth+1):
                    kept_any = True
    return kept_any

# navigate folders to find relevant files, then extract passages from them. a HARD file cap bounds how many
# distinct files we are allowed to reference: lookup -> 1 (single-file lock), synthesis -> MAX_FILES.
# navigate folders to find relevant files, then extract passages from them. a HARD file cap bounds how many
# distinct files we are allowed to reference: lookup -> 1 (single-file lock), synthesis -> MAX_FILES.
def greedy_teleport_search(query, qtype, counter, only_files=None):
    import heapq
    cap = LOOKUP_FILES if qtype=="lookup" else MAX_FILES
    if only_files: cap = 1   # question named a specific file -> single-file lock
    pq=[]; heapq.heappush(pq,(-1.0,0,ROOT,ROOT.name,0))
    nc=0; evidence=[]; visited=set(); files=[]; stats={"nodes":0,"docs":0}
    print(f"\n{'#'*78}\nGREEDY SEARCH  (query type: {qtype})  —  hard file cap = {cap}\n{'#'*78}")
    while pq and stats["nodes"]<BUDGET:
        neg,_,node,path,depth=heapq.heappop(pq); score=-neg
        stats["nodes"]+=1
        print(f"\n{'  '*depth}→ EXAMINING: {path}  (priority {score:.2f})")

        if node.node_type in ("document","section"):         # a file: commit and extract its relevant passages
            fkey0=node.metadata.get('source_file') or node.path or node.name
            if only_files and not any(lk in fkey0 for lk in only_files):
                print(f"{'  '*depth}  ⦻ off-lock file '{node.name}' — skipped (locked to named file)"); continue
            stats["docs"]+=1
            print(f"{'  '*depth}  ⊙ COMMIT to file '{node.name}'")
            before=len(evidence)
            mine_unit(node, query, qtype, counter, evidence, depth)
            if len(evidence)>before:                      # one commit == one file; key it off the committed node
                fkey=node.metadata.get('source_file') or node.path or node.name
                if fkey not in files: files.append(fkey)
                print(f"{'  '*depth}  ✓ kept {len(evidence)-before} passage(s); files used {len(files)}/{cap}")
                if len(files)>=cap:
                    print(f"{'  '*depth}  🔒 hit the file cap ({cap}) — search stops, no more files.")
                    break
                continue                                      # synthesis with room left — look for the next file
            print(f"{'  '*depth}  ✗ nothing relevant inside — continuing to look")
            continue

        kids=node.children
        if not kids:                                         # stray chunk in a folder
            keep,_=leaf_decision(query,node,qtype,counter)
            if keep:
                evidence.append(_tag_rel(node, KEEP_LEAF_AT))
                f=node.metadata.get('source_file') or node.path or node.name
                if f not in files: files.append(f)
                print(f"{'  '*depth}  ✓ kept stray passage; files used {len(files)}/{cap}")
                if len(files)>=cap:
                    print(f"{'  '*depth}  🔒 hit the file cap ({cap}) — search stops."); break
            continue

        print(f"{'  '*depth}  ≡ scoring {len(kids)} children and queueing them")
        scores,reasons,oks=robust_score(query,node,kids,counter)

        # --- NEW: Folder Summary Short-Circuit ---
        if qtype == "synthesis":
            candidate_summaries = []
            for i in range(len(kids)):
                if scores[i] >= RELEVANT_AT:
                    child = kids[i]
                    # Promote the summary into the content field so downstream handling treats it like a leaf chunk
                    mock_content = f"{child.name}: {child.summary}"
                    md = dict(child.metadata)
                    md["source_file"] = node.metadata.get('source_file') or node.path or node.name
                    md["_rel"] = scores[i]
                    candidate_summaries.append(replace(child, content=mock_content, metadata=md))
    
            if candidate_summaries:
                print(f"{'  '*depth}  [?] checking if {len(candidate_summaries)} relevant sub-summaries in '{node.name}' fully answer the query...")
                if is_complete_folder(query, candidate_summaries, counter):
                    print(f"{'  '*depth}  🎯 YES! Folder summaries contain the complete answer. Halting search.")
                    evidence.extend(candidate_summaries)
                    fkey = node.metadata.get('source_file') or node.path or node.name
                    if fkey not in files: files.append(fkey)
                    break # immediately end the output/search
                else:
                    print(f"{'  '*depth}  · summaries alone are incomplete; continuing descent.")
        # -----------------------------------------

        for i in sorted(range(len(kids)),key=lambda i:-scores[i]):
            child=kids[i]
            if child.node_id in visited: continue
            visited.add(child.node_id); nc+=1
            heapq.heappush(pq,(-scores[i],nc,child,path+" > "+child.name,depth+1))
            mark="★" if scores[i]>=0.8 else ("✓" if scores[i]>=RELEVANT_AT else "·")
            print(f"{'  '*depth}    {mark} {scores[i]:.2f} {clip(child.name,48)}"
                  + (f"  — {reasons[i]}" if reasons[i] else ""))
        front=sorted(pq)[:8]
        if front:
            print(f"{'  '*depth}  ⋯ global frontier (top {len(front)}): "
                  + " | ".join(f"{-s:.2f} {p.split(' > ')[-1][:24]}" for s,_,_,p,_ in front))

    print(f"\n{'#'*78}\nSEARCH COMPLETE — navigated {stats['nodes']} nodes, committed to {stats['docs']} file(s), "
          f"kept {len(evidence)} passage(s) across {len(files)} file(s) (cap {cap})")
    return evidence

counter_for_kids=Counters()
print("greedy search ready: passage-level extraction + hard file cap (lookup=1, synthesis=MAX_FILES)")

greedy search ready: passage-level extraction + hard file cap (lookup=1, synthesis=MAX_FILES)


## Load the corpus tree

In [6]:
if not TREE_FILE.exists():
    raise SystemExit(f"tree not found at {TREE_FILE.resolve()}; build it first with prototype 5")
ROOT = TreeNode.from_dict(json.loads(TREE_FILE.read_text(encoding="utf-8")))
index_tree(ROOT)
print(f"loaded tree {ROOT.name}; {ROOT.count_leaves()} leaves and {len(ROOT.children)} top level children")

loaded tree folders; 95458 leaves and 10 top level children


## Pipeline functions

In [7]:

import ast
from tqdm import tqdm   # plain text bar; no ipywidgets needed

# ---------- load questions (json; qid/stem/options/answer(s)/difficulty) ----------
def load_questions_json(path):
    raw=Path(path).read_text(encoding="utf-8")
    try: data=json.loads(raw)
    except Exception: data=ast.literal_eval(raw)
    if isinstance(data,dict): data=data.get("questions",list(data.values()))
    out=[]; skipped=[]
    for i,d in enumerate(data,1):
        qid=str(d.get("qid") or d.get("id") or f"q{i}")
        stem=str(d.get("stem") or d.get("question") or "").strip()
        opts={str(k).upper():str(v).strip() for k,v in (d.get("options") or {}).items() if str(v).strip()}
        ansv=d.get("answers") or ([d.get("answer")] if d.get("answer") else [])
        letters={m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Fa-f])(?![A-Za-z])"," ".join(map(str,ansv)))}
        ans=[L for L in "ABCDEF" if L in letters]
        if not stem: skipped.append((qid,"no stem")); continue
        if len(opts)<2: skipped.append((qid,"<2 options")); continue
        if not ans: skipped.append((qid,"no answer letter")); continue
        out.append(Question(qid, stem, opts, ans[0], str(d.get("difficulty") or ""), ans))
    return out, skipped

# ---------- step 2: answer one question with the newest_prototype_3 TreeRAG pipeline ----------
def _commit_keys(node, acc):
    if node.node_type in ("document","section"):
        acc.add(node.metadata.get('source_file') or node.path or node.name)
    for ch in node.children: _commit_keys(ch, acc)
    return acc

def _file_lock(q):
    stem_low=q.stem.lower(); lock=set()
    for k in _commit_keys(ROOT, set()):
        base=k.replace('\\','/').split('/')[-1]
        base=re.sub(r'\.(docx|doc|pdf|xlsx|pptx|txt)$','',base,flags=re.I).strip()
        if len(base)>=6 and base.lower() in stem_low: lock.add(k)
    return lock or None

def answer_one(q):
    c=Counters(); t0=time.perf_counter()
    qtype=classify_query(q, c)
    evidence=greedy_teleport_search(q.stem, qtype, c, only_files=_file_lock(q))
    response, quotes, working_doc, contributions = build_running_doc(q, evidence, c)
    dt=time.perf_counter()-t0
    kept=[k for k in contributions if k.get('answered')]
    files=[]
    for k in kept:
        s=k.get('src')
        if s and s not in files: files.append(s)
    if not files:
        for e in evidence:
            s=e.metadata.get("source_file") or e.path or e.name
            if s and s not in files: files.append(s)
    return {"id":q.qid,"question":q.stem,"options":q.options,"correct_answers":_ans_list(q),
            "treerag_response":response,"files_referenced":files,"qtype":qtype,
            "leaves":len(evidence),"time_sec":round(dt,3),
            "in_tokens":c.in_tok,"out_tokens":c.out_tok}

# crash-safe answer store
def _ans_load():
    return json.loads(ANS_STORE.read_text()) if ANS_STORE.exists() else {}
def _ans_save(store):
    tmp=ANS_STORE.with_suffix(".tmp"); tmp.write_text(json.dumps(store)); tmp.replace(ANS_STORE)

def _atomic_write(path, text):
    p=Path(path); tmp=p.with_suffix(p.suffix+".tmp")
    tmp.write_text(text); tmp.replace(p)               # write-then-rename so a crash never leaves a half file

def _write_answers(store, questions):
    records=[store[q.qid] for q in questions if q.qid in store]   # keep original order; only what's done so far
    _atomic_write(ANSWERS_FILE, json.dumps(records,indent=2))
    return records

def run_answers(questions, bar):
    store=_ans_load()
    _write_answers(store, questions)                   # refresh output file up front (covers resume)
    for q in questions:
        if q.qid not in store:
            store[q.qid]=answer_one(q)
            _ans_save(store)                           # per-question cache checkpoint
            _write_answers(store, questions)           # AND refresh treerag_answers_2.json every question
        bar.update(1)
    return _write_answers(store, questions)

# ---------- benchmark loaders/judge ----------
def _read(path):
    raw=Path(path).read_text(encoding="utf-8")
    try: data=json.loads(raw)
    except Exception: data=ast.literal_eval(raw)
    if isinstance(data,dict): data=data.get("questions",list(data.values()))
    return data

def _rec_answers(d):
    a=d.get("correct_answers") or d.get("answers") or ([d.get("answer")] if d.get("answer") else [])
    letters={m.upper() for m in re.findall(r"(?<![A-Za-z])([A-Fa-f])(?![A-Za-z])"," ".join(map(str,a)))}
    return [L for L in "ABCDEF" if L in letters]

def _norm(d, response_key, time_key, in_key, out_key):
    return {"id":str(d.get("id") or d.get("qid")),
            "question":d.get("question") or d.get("stem") or "",
            "options":{str(k).upper():str(v) for k,v in (d.get("options") or {}).items()},
            "correct":_rec_answers(d),
            "qtype":d.get("qtype",""),
            "response":str(d.get(response_key) or ""),
            "time":float(d.get(time_key) or 0.0),
            "in":int(d.get(in_key) or 0),
            "out":int(d.get(out_key) or 0)}

def _parse_judge(text):
    t=(text or "").strip(); m=re.search(r'\{.*\}', t, re.S)
    if m:
        try:
            o=json.loads(m.group(0)); return max(0.0,min(1.0,float(o.get("score",0)))), str(o.get("reason",""))[:300]
        except Exception: pass
    m=re.search(r'(\d?\.\d+|\d)', t)
    return (max(0.0,min(1.0,float(m.group(1)))) if m else 0.0), "unparsed judge reply"

def judge_accuracy(question, options, correct, response):
    multi=len(correct)>1
    opts="\n".join(f"{L}. {options[L]}" for L in "ABCDEF" if L in options)
    gold="\n".join(f"{L}. {options.get(L,'')}" for L in correct)
    prompt=("you are a fair grader. a system answered an open question in its own words and could not see the "
            "choices. the multiple choice version below has the correct option(s) marked and those are the ground "
            "truth. the correct answer may be ONE OR MORE options.\n\n"
            f"QUESTION: {question}\nOPTIONS:\n{opts}\nCORRECT OPTION(S): {', '.join(correct)}\n{gold}\n\n"
            f"SYSTEM RESPONSE:\n{response or '(empty)'}\n\n"
            "grade from 0.0 to 1.0 how well the response matches the MEANING of the correct option(s). judge by "
            "meaning not wording. "
            +("when several options are correct, give full credit only if the response conveys ALL of them, and "
              "proportional partial credit for covering some. " if multi else
              "give full or near full credit when it conveys the correct idea even in different words, partial when "
              "incomplete. ")
            +"give low credit when it matches a wrong option or is irrelevant. reply with ONLY json:\n"
            '{"score": <0.0 to 1.0>, "reason": "<one concise sentence>"}')
    return _parse_judge(ask(prompt,num_predict=400))

print("combined pipeline functions ready")


combined pipeline functions ready


## Run all three steps (single progress bar)

In [8]:

# ============ RUN EVERYTHING — SINGLE PROGRESS BAR, QUIET ============
import contextlib, io, sys

questions_all, skipped = load_questions_json(QUESTIONS_FILE)
questions = questions_all[:MAX_QUESTIONS]

qms_raw=_read(QMS_FILE)
QMS={r["id"]:r for r in (_norm(d,"answer","elapsed_sec","input_tokens","output_tokens") for d in qms_raw)}
answer_ids=[q.qid for q in questions]
common_pre=[i for i in answer_ids if i in QMS]

N=len(questions); C=len(common_pre)
_store=_ans_load(); done_ans=sum(1 for q in questions if q.qid in _store)
JCACHE_FILE=OUT_DIR/"judge_cache.json"; JCACHE=json.loads(JCACHE_FILE.read_text()) if JCACHE_FILE.exists() else {}
CLS_FILE=OUT_DIR/"qtype_cache.json";  CLS =json.loads(CLS_FILE.read_text())  if CLS_FILE.exists()  else {}
def _jsave():
    t=JCACHE_FILE.with_suffix(".tmp"); t.write_text(json.dumps(JCACHE)); t.replace(JCACHE_FILE)
def _csave():
    t=CLS_FILE.with_suffix(".tmp"); t.write_text(json.dumps(CLS)); t.replace(CLS_FILE)

done_judge=sum(1 for i in common_pre for s in ("tree","qms") if f"{s}::{i}" in JCACHE)
done_cls=sum(1 for i in common_pre if i in CLS)
TOTAL=N + 2*C + C
DONE0=done_ans + done_judge + done_cls

_null=io.StringIO()   # swallow any prints from inner functions so only the bar shows
MASTER=tqdm(total=TOTAL, initial=DONE0, desc="TreeRAG → judge → benchmark", dynamic_ncols=True, leave=True)

with contextlib.redirect_stdout(_null):
    # phase 1+2: answer first 100 -> treerag_answers_2.json
    records=run_answers(questions, MASTER)
    tree_raw=_read(ANSWERS_FILE)
    TREE={r["id"]:r for r in (_norm(d,"treerag_response","time_sec","in_tokens","out_tokens") for d in tree_raw)}
    ids=[i for i in TREE if i in QMS]
    only_tree=[i for i in TREE if i not in QMS]; only_qms=[i for i in QMS if i not in TREE]

    # -------- incremental report writer: rebuild from current caches and atomically write all outputs --------
    def _flush_reports(note_map=None):
        note_map=note_map or {}
        rws=[]
        for i in ids:
            if f"tree::{i}" not in JCACHE or f"qms::{i}" not in JCACHE: continue
            t,q=TREE[i],QMS[i]
            ts=JCACHE[f"tree::{i}"]["score"]; qs=JCACHE[f"qms::{i}"]["score"]
            rws.append({"id":i,"qtype":CLS.get(i,""),"question":t["question"][:90],
                "tree_accuracy":ts,"qms_accuracy":qs,"accuracy_delta":round(ts-qs,3),
                "tree_time":t["time"],"qms_time":q["time"],"tree_in":t["in"],"qms_in":q["in"],
                "tree_out":t["out"],"qms_out":q["out"],
                "tree_reason":JCACHE[f"tree::{i}"]["reason"],"qms_reason":JCACHE[f"qms::{i}"]["reason"],
                "explanation":note_map.get(i,"")})
        pq=pd.DataFrame(rws)
        if pq.empty: return pq,{},{}
        def _ms2(x):
            x=list(map(float,x))
            return (round(float(np.mean(x)),3), round(float(np.std(x,ddof=1)) if len(x)>1 else 0.0,3)) if x else (0.0,0.0)
        mets={"accuracy":("tree_accuracy","qms_accuracy"),"time_sec":("tree_time","qms_time"),
              "input_tokens":("tree_in","qms_in"),"output_tokens":("tree_out","qms_out")}
        summ={}
        for name,(tc,qc) in mets.items():
            tm,tsd=_ms2(pq[tc]); qm,qsd=_ms2(pq[qc])
            summ[name]={"treerag_mean":tm,"treerag_std":tsd,"qms_mean":qm,"qms_std":qsd,
                        "mean_diff_tree_minus_qms":round(tm-qm,3)}
        bt={}
        for qt,grp in pq.groupby("qtype"):
            e={"n":int(len(grp))}
            for name,(tc,qc) in mets.items():
                tm,tsd=_ms2(grp[tc]); qm,qsd=_ms2(grp[qc])
                e[name]={"treerag_mean":tm,"treerag_std":tsd,"qms_mean":qm,"qms_std":qsd,
                         "mean_diff_tree_minus_qms":round(tm-qm,3)}
            bt[qt or "unlabeled"]=e
        rep={"systems":["treerag","qms"],"n_questions":int(len(pq)),
             "only_in_treerag":only_tree,"only_in_qms":only_qms,"summary":summ,
             "flagged":{i:note_map[i] for i in note_map},"per_question":pq.to_dict(orient="records")}
        _atomic_write(REPORT_JSON, json.dumps(rep,indent=2))
        _atomic_write(BYTYPE_JSON, json.dumps({"n_questions":int(len(pq)),"by_type":bt},indent=2))
        _atomic_write(REPORT_CSV, pq.to_csv(index=False))
        return pq,summ,bt

    # phase 3a: judge both systems
    for i in ids:
        for sysname,store in (("tree",TREE),("qms",QMS)):
            ck=f"{sysname}::{i}"
            if ck in JCACHE: continue
            r=store[i]
            sc,why=judge_accuracy(r["question"], r["options"] or (TREE.get(i) or QMS.get(i))["options"],
                                  r["correct"] or (TREE.get(i) or QMS.get(i))["correct"], r["response"])
            JCACHE[ck]={"score":sc,"reason":why}; _jsave(); _flush_reports(); MASTER.update(1)

    # phase 3b: classify lookup vs synthesis
    for i in ids:
        if i in CLS: continue
        t=TREE[i]
        qt=t.get("qtype") or classify_query(Question(i,t["question"],t["options"],
                                                     (t["correct"][0] if t["correct"] else ""),"",t["correct"]), Counters())
        CLS[i]=qt; _csave(); _flush_reports(); MASTER.update(1)

    # everything to here is cached; build the current tables from caches
    per_q, summary, by_type = _flush_reports()

    # phase 4: explain flagged (large accuracy gap or metric outlier), writing after each note
    def _anoms(col):
        v=per_q[col].astype(float); mu,sd=v.mean(),(v.std(ddof=1) or 1.0)
        return set(per_q.loc[(v-mu).abs()>3*sd,"id"])
    flag_metric=set().union(*[_anoms(c) for c in ["tree_time","qms_time","tree_in","qms_in","tree_out","qms_out"]]) if len(per_q)>1 else set()
    flag_acc=set(per_q.loc[per_q["accuracy_delta"].abs()>=ACC_GAP_FLAG,"id"]) if len(per_q) else set()
    flagged=[i for i in ids if i in flag_acc or i in flag_metric]

    NOTES_FILE=OUT_DIR/"explain_cache.json"; notes=json.loads(NOTES_FILE.read_text()) if NOTES_FILE.exists() else {}
    todo_flag=[i for i in flagged if i not in notes]
    MASTER.total += len(todo_flag); MASTER.refresh()
    def explain(i):
        t,q=TREE[i],QMS[i]; r=per_q.set_index("id").loc[i]
        prompt=("two retrieval systems answered the same question and their results differ. in 1-2 sentences say why, "
                "referring to the responses and the numbers. be concrete and neutral.\n\n"
                f"QUESTION: {t['question']}\nCORRECT: {', '.join(t['correct'])}\n\n"
                f"TREERAG response: {t['response'][:600]}\nTREERAG accuracy {r['tree_accuracy']}, time {r['tree_time']}s, "
                f"in {r['tree_in']}, out {r['tree_out']}\n\n"
                f"QMS response: {q['response'][:600]}\nQMS accuracy {r['qms_accuracy']}, time {r['qms_time']}s, "
                f"in {r['qms_in']}, out {r['qms_out']}\n\nexplanation:")
        return ask(prompt,num_predict=220,think=False).strip()
    for i in todo_flag:
        notes[i]=explain(i)
        _atomic_write(NOTES_FILE, json.dumps(notes))
        _flush_reports(notes)                      # keep report current with explanations
        MASTER.update(1)

    per_q, summary, by_type = _flush_reports(notes)   # final authoritative write

MASTER.close()

# --- the only visible output: a compact summary ---
print(f"done — answered {len(questions)}, judged {len(ids)} common questions")
print(f"wrote {REPORT_JSON}")
print(f"wrote {BYTYPE_JSON}")
print(f"wrote {REPORT_CSV}\n")
print(pd.DataFrame(summary).T[["treerag_mean","treerag_std","qms_mean","qms_std","mean_diff_tree_minus_qms"]].to_string())
print("\nby type (accuracy):")
for qt,e in by_type.items():
    print(f"  {qt:12s} n={e['n']:<4d} tree={e['accuracy']['treerag_mean']:.3f}  qms={e['accuracy']['qms_mean']:.3f}  delta={e['accuracy']['mean_diff_tree_minus_qms']:+.3f}")


TreeRAG → judge → benchmark: 97it [04:16,  3.28s/it]                                                                                                              

done — answered 19, judged 19 common questions
wrote second_benchmark/benchmark_report_2.json
wrote second_benchmark/benchmark_by_type_2.json
wrote second_benchmark/benchmark_per_question_2.csv

               treerag_mean  treerag_std  qms_mean  qms_std  mean_diff_tree_minus_qms
accuracy              0.622        0.456     0.748    0.387                    -0.126
time_sec            416.895      124.656    34.048   16.268                   382.847
input_tokens      43780.053    17698.748   471.105    4.040                 43308.948
output_tokens     20491.368     5502.812    58.474    8.959                 20432.894

by type (accuracy):
  lookup       n=19   tree=0.622  qms=0.748  delta=-0.126
